EE 344 Final Project - Data Preprocessing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Mounted at /content/drive


In [2]:
import pandas as pd
import glob

DATA_FOLDER = "/content/drive/MyDrive/Junior Year/EE 344/Final Project Data/"

# Read all CSVs
file_paths = sorted(glob.glob(DATA_FOLDER + "*.csv"))
all_data = pd.concat([pd.read_csv(fp) for fp in file_paths], ignore_index=True)

# Drop columns that cause leakage
columns_to_keep = [
    'FL_DATE', 'OP_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'DEP_DELAY'
]
all_data = all_data[columns_to_keep]

# Convert date column
all_data['FL_DATE'] = pd.to_datetime(all_data['FL_DATE'])

# Chronological split: 2014-2018 train, 2019 test
train_data = all_data[all_data['FL_DATE'].dt.year <= 2018].copy()
test_data = all_data[all_data['FL_DATE'].dt.year == 2019].copy()

print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")

ValueError: No objects to concatenate

In [ ]:
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.preprocessing import StandardScaler

# ------------------------
# Preprocessing Pipeline
# ------------------------
# Used for all models. Depending on model needs, only certain columns will be chosen for each model's data.

def preprocess_flight_data(df):
    df = df.copy()

    # Ensure FL_DATE is datetime
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

    # Extract basic time features from CRS_DEP_TIME
    df['CRS_DEP_TIME'] = df['CRS_DEP_TIME'].fillna(0).astype(int)
    df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100  # 0-23 hour
    df['DEP_MIN'] = df['CRS_DEP_TIME'] % 100    # minutes if needed
    df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek  # Monday=0, Sunday=6
    df['MONTH'] = df['FL_DATE'].dt.month

    # Time-of-day label
    def get_time_of_day(hour):
        if 5 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        else:
            return 'night'
    df['TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

    # Holiday flag using USFederalHolidayCalendar
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['FL_DATE'].min(), end=df['FL_DATE'].max())
    df['IS_HOLIDAY'] = df['FL_DATE'].isin(holidays).astype(int)

    # Cyclic encoding placeholders (for linear models)
    df['DEP_HOUR_SIN'] = np.sin(2 * np.pi * df['DEP_HOUR']/24)
    df['DEP_HOUR_COS'] = np.cos(2 * np.pi * df['DEP_HOUR']/24)
    df['DAY_OF_WEEK_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK']/7)
    df['DAY_OF_WEEK_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK']/7)

    return df

# Apply preprocessing to all data.
train_data = preprocess_flight_data(train_data)
test_data = preprocess_flight_data(test_data)

print(train_data.head())

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# -------------------------
# Feature / Target Setup
# -------------------------
# Selects the columns for each model.

# Categorical columns for one-hot (linear models)
categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK']

# Numeric columns
numeric_cols = ['DISTANCE', 'CRS_ELAPSED_TIME']

# Cyclic columns (for linear models)
cyclic_cols = ['DEP_HOUR_SIN', 'DEP_HOUR_COS', 'DAY_OF_WEEK_SIN', 'DAY_OF_WEEK_COS']

# Tree model features (use raw hour/day/month/time-of-day + categorical)
tree_categorical_cols = ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_WEEK', 'TIME_OF_DAY']
tree_numeric_cols = ['DISTANCE', 'CRS_ELAPSED_TIME']

# Seperating the columns and preparing the data for each respective model based on approach.
def prepare_model_data(df, approach='binary', model_type='linear'):
    df = df.copy()

    if approach == 'binary':
        # Delayed or not (DEP_DELAY > 15)
        df['TARGET'] = (df['DEP_DELAY'] > 15).astype(int)
    elif approach == 'regression':
        # How much delay
        df['TARGET'] = df['DEP_DELAY']  # continuous
    elif approach == 'multiclass':
        # Predict delay bin
        bins = [0,15,30,60,np.inf]
        labels = [0,1,2,3]  # Bin Number
        df['TARGET'] = pd.cut(df['DEP_DELAY'], bins=bins, labels=labels, right=True)
        df['TARGET'] = df['TARGET'].astype(int)
    else:
        raise ValueError("Approach must be 'binary', 'regression', or 'multiclass'")

    # Linear Model Processing
    if model_type == 'linear':
        # One-hot encode categorical
        df_ohe = pd.get_dummies(df[categorical_cols], drop_first=True)
        # Scale numeric
        scaler = StandardScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), columns=numeric_cols, index=df.index)
        # Combine numeric + one-hot + cyclic
        X = pd.concat([df_scaled, df_ohe, df[cyclic_cols]], axis=1)
    # Tree Model Processing
    elif model_type == 'tree':
        # One-hot encode tree categorical columns
        df_ohe = pd.get_dummies(df[tree_categorical_cols], drop_first=True)
        # Combine numeric + one-hot
        X = pd.concat([df[tree_numeric_cols], df_ohe], axis=1)
    else:
        raise ValueError("Model type must be 'linear' or 'tree'")

    y = df['TARGET']
    return X, y

# Linear model datasets
X_train_linear_bin, y_train_bin = prepare_model_data(train_data, approach='binary', model_type='linear')
X_test_linear_bin, y_test_bin = prepare_model_data(test_data, approach='binary', model_type='linear')

X_train_linear_reg, y_train_reg = prepare_model_data(train_data, approach='regression', model_type='linear')
X_test_linear_reg, y_test_reg = prepare_model_data(test_data, approach='regression', model_type='linear')

X_train_linear_multi, y_train_multi = prepare_model_data(train_data, approach='multiclass', model_type='linear')
X_test_linear_multi, y_test_multi = prepare_model_data(test_data, approach='multiclass', model_type='linear')

# Tree model datasets
X_train_tree_bin, y_train_bin_tree = prepare_model_data(train_data, approach='binary', model_type='tree')
X_test_tree_bin, y_test_bin_tree = prepare_model_data(test_data, approach='binary', model_type='tree')

X_train_tree_reg, y_train_reg_tree = prepare_model_data(train_data, approach='regression', model_type='tree')
X_test_tree_reg, y_test_reg_tree = prepare_model_data(test_data, approach='regression', model_type='tree')

X_train_tree_multi, y_train_multi_tree = prepare_model_data(train_data, approach='multiclass', model_type='tree')
X_test_tree_multi, y_test_multi_tree = prepare_model_data(test_data, approach='multiclass', model_type='tree')

print("Datasets ready!")